In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import sys
sys.path.append("..")  # helpers.py и prompt_evaluator.py лежат в корне курса
from helpers import add_user_message, chat
from prompt_evaluator import PromptEvaluator

client = Anthropic()
model = "claude-haiku-5-5"

In [ ]:
# Больше потоков — быстрее, но можно получить ошибку rate limit
evaluator = PromptEvaluator(client, model, max_concurrent_tasks=3)

In [ ]:
dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan "
    "for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    output_file="15_dataset.json",
    num_cases=3,
)

In [ ]:
def run_prompt(prompt_inputs):
    """Первая, специально простая версия prompt"""
    prompt = f"""
What should this person eat?

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(client, model, messages)

In [ ]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="15_dataset.json",
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
""",
    json_output_file="15_output.json",
    html_output_file="15_output.html",
)